# wavexplain: run on your own data

Take a sales CSV, train a forecaster on your own series, and get interpretable
**driver cards** that split each forecast into named parts that sum exactly to
the prediction: a typical-pattern base, a recent-trend effect, and a promotion
effect.

**Why training happens on your data.** The model has a per-series embedding
table, so series it has never seen are new to it and it cannot forecast them
zero-shot. Each run trains from scratch on the series you provide. This is quick
on the free GPU.

**What this is and isn't.** It is research-grade tooling being made usable. It
has not been shown to reduce real-world waste at a retailer, so treat the numbers
as a decision aid, not a guarantee. The attribution is **counterfactual, not
causal**: contributions come from revealing the real input on top of a neutral
baseline, not from a causal model of demand. The promotion bucket is the
least-validated part of the decomposition, so the `Promotion-driven` badge is
provisional, not a confident claim.

> Set **Runtime > Change runtime type > GPU** before running. To smoke-test on a
> fresh runtime with no upload, leave `USE_BUILT_IN_SAMPLE = True` in step 1.


In [ ]:
# 1. Environment + install the package
import subprocess, sys
import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("No GPU detected. Runtime > Change runtime type > GPU. "
          "The built-in sample still runs on CPU, just slower.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/kesjien/wavexplain.git"],
    check=True,
)
print("Installed wavexplain.")


In [ ]:
# 2. Imports
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import OrderedDict
from torch.utils.data import Dataset, DataLoader, Subset

from wavexplain import MultiSeriesWaveNet, CounterfactualExplainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
# This is just execution statistics tracked via Google Analytics
from google.colab import userdata
import requests
import uuid

def track_notebook_execution():
    measurement_id = "G-98HEZ541RF"
    
    try:
        api_secret = userdata.get('WaveXplain_Colab')
    except Exception:
        api_secret = None
        
    if not api_secret:
        return

    client_id = str(uuid.uuid4())
    url = f"https://www.google-analytics.com/mp/collect?measurement_id={measurement_id}&api_secret={api_secret}"
    
    payload = {
        "client_id": client_id,
        "events": [{
            "name": "notebook_executed",
            "params": {"notebook_name": "run_on_your_data"}
        }]
    }
    
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception:
        pass

track_notebook_execution()

In [ ]:
# 3. Config (matches the trained model's I/O contract)
INPUT_LENGTH   = 90     # days of history the model reads
HORIZON        = 16     # days the model predicts
REPORT_HORIZON = 16     # days summed for the headline number and the buckets (<= HORIZON)
RECENT_DAYS    = 14     # trailing window counted as "recent trend" (non-promo days)
HISTORY_PLOTTED = 16    # recent actual days drawn on the chart

EPOCHS        = 60
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
MIN_HISTORY   = INPUT_LENGTH + HORIZON   # a series needs at least this many days
MAX_CARDS     = 12      # how many series to build cards for
SEED          = 0

torch.manual_seed(SEED); np.random.seed(SEED)
assert REPORT_HORIZON <= HORIZON


## Step 1: get your data in

Leave `USE_BUILT_IN_SAMPLE = True` to run everything on a synthetic panel with no
upload. Set it to `False` to upload a long-format CSV with, at minimum, a **date**
column and a **sales** column, plus a series identifier (a single `series_id`
column, or both `store` and `item`). An `onpromotion` column is optional; without
it the promotion channel is all zeros and the promo bucket will be ~0.


In [ ]:
# 4. Data input: built-in sample OR your CSV
USE_BUILT_IN_SAMPLE = True   # set False to upload your own CSV

if USE_BUILT_IN_SAMPLE:
    # Synthetic panel with a weekly cycle, a slow trend, and clear promo spikes.
    rng = np.random.default_rng(SEED)
    n_series, n_days = 8, 400
    dates = pd.date_range("2015-01-01", periods=n_days, freq="D")
    rows = []
    for s in range(n_series):
        level  = rng.uniform(8, 30)
        trend  = np.linspace(0, rng.uniform(-5, 10), n_days)
        weekly = 0.3 * level * np.sin(2 * np.pi * (np.arange(n_days) % 7) / 7 + rng.uniform(0, 6))
        promo  = (rng.random(n_days) < 0.08).astype(int)
        noise  = rng.normal(0, 0.10 * level, n_days)
        sales  = np.clip(level + trend + weekly + promo * rng.uniform(0.8, 1.6) * level + noise, 0, None)
        for d in range(n_days):
            rows.append((dates[d], f"S{s:02d}", round(float(sales[d]), 2), int(promo[d])))
    df = pd.DataFrame(rows, columns=["date", "series_id", "sales", "onpromotion"])
    print("Built-in sample:", df.series_id.nunique(), "series,", len(df), "rows")
else:
    from google.colab import files
    up = files.upload()
    fname = next(iter(up))
    df = pd.read_csv(fname)
    print("Loaded", fname, "shape", df.shape)

df.head()


In [ ]:
# 5. Normalize columns to: date, series_id, sales, onpromotion
def normalize_columns(df):
    df = df.copy()
    cols = {c.lower().strip(): c for c in df.columns}
    def pick(*names):
        for n in names:
            if n in cols:
                return cols[n]
        return None

    date_c  = pick("date", "ds", "day")
    sales_c = pick("sales", "unit_sales", "y", "quantity", "qty", "units")
    promo_c = pick("onpromotion", "on_promotion", "promotion", "promo", "is_promo")
    sid_c   = pick("series_id", "series", "id")
    store_c = pick("store", "store_nbr", "store_id")
    item_c  = pick("item", "item_nbr", "item_id", "sku", "product")

    if date_c is None or sales_c is None:
        raise ValueError("CSV needs at least a date column and a sales column. "
                         "Found: " + ", ".join(df.columns))

    out = pd.DataFrame()
    out["date"]  = pd.to_datetime(df[date_c])
    out["sales"] = pd.to_numeric(df[sales_c], errors="coerce").fillna(0.0).clip(lower=0)
    if sid_c is not None:
        out["series_id"] = df[sid_c].astype(str)
    elif store_c is not None and item_c is not None:
        out["series_id"] = df[store_c].astype(str) + "_" + df[item_c].astype(str)
    else:
        raise ValueError("Need a series identifier: a 'series_id' column, "
                         "or both 'store' and 'item' columns.")
    if promo_c is not None:
        out["onpromotion"] = (pd.to_numeric(df[promo_c], errors="coerce").fillna(0) > 0).astype(float)
    else:
        out["onpromotion"] = 0.0
    return out

panel = normalize_columns(df)
print("Series:", panel.series_id.nunique(),
      "| Dates:", panel.date.min().date(), "->", panel.date.max().date())
panel.head()


In [ ]:
# 6. Dense per-series daily arrays (sales + promo), sharing each series' own
# date range. Missing days are filled with 0 sales / not-on-promotion.
def build_series_arrays(panel, min_history=MIN_HISTORY):
    series = {}
    for sid, g in panel.groupby("series_id"):
        g = g.sort_values("date")
        idx = pd.date_range(g.date.min(), g.date.max(), freq="D")
        g = g.set_index("date").reindex(idx)
        sales = g["sales"].fillna(0.0).to_numpy(dtype=np.float32)
        promo = g["onpromotion"].fillna(0.0).to_numpy(dtype=np.float32)
        if len(sales) >= min_history:
            series[sid] = {"sales": sales, "promo": promo}
    if not series:
        raise ValueError(f"No series has at least {min_history} days "
                         f"({INPUT_LENGTH} input + {HORIZON} target). "
                         "Provide longer histories, or lower INPUT_LENGTH / HORIZON.")
    series_ids = sorted(series.keys())
    id_to_index = {sid: i for i, sid in enumerate(series_ids)}
    print(f"Kept {len(series_ids)} series with >= {min_history} days.")
    return series, series_ids, id_to_index

series, series_ids, id_to_index = build_series_arrays(panel)
NUM_SERIES = len(series_ids)


In [ ]:
# 7. Shared input transform + windowed dataset.
# The model was trained on log1p(sales) for channel 0 and RAW 0/1 promo for
# channel 1. make_log_input is the single source of truth for that transform,
# used by both training and card-building so the two never drift.
def make_log_input(sales_window, promo_window):
    log_sales = np.log1p(np.clip(sales_window, 0, None))
    stacked = np.stack([log_sales, promo_window], axis=0).astype("float32")  # (2, L)
    return torch.from_numpy(stacked)

class WindowDataset(Dataset):
    """Sliding windows. x = make_log_input(sales, promo) -> (2, L);
    target = RAW sales over the next HORIZON days (log1p is applied in the loss)."""
    def __init__(self, series, id_to_index, input_length=INPUT_LENGTH,
                 horizon=HORIZON, stride=1):
        self.series = series
        self.id_to_index = id_to_index
        self.input_length = input_length
        self.horizon = horizon
        self.samples = []
        for sid, arr in series.items():
            n = len(arr["sales"])
            for start in range(0, n - input_length - horizon + 1, stride):
                self.samples.append((sid, start))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        sid, start = self.samples[i]
        arr = self.series[sid]
        s, e = start, start + self.input_length
        x = make_log_input(arr["sales"][s:e], arr["promo"][s:e])            # (2, L)
        y = torch.from_numpy(arr["sales"][e:e + self.horizon].astype("float32"))  # raw
        sid_t = torch.tensor(self.id_to_index[sid], dtype=torch.long)
        return x, sid_t, y

full_ds = WindowDataset(series, id_to_index)

# Time-based split: the last window of each series is held out for validation,
# matching the paper's setup (final HORIZON days per series as the holdout).
last_by_series = {}
for i, (sid, start) in enumerate(full_ds.samples):
    last_by_series[sid] = i
val_idx = sorted(set(last_by_series.values()))
val_set = set(val_idx)
train_idx = [i for i in range(len(full_ds)) if i not in val_set]

train_loader = DataLoader(Subset(full_ds, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(Subset(full_ds, val_idx),   batch_size=BATCH_SIZE, shuffle=False)
print(f"Train windows: {len(train_idx)} | Val windows: {len(val_idx)}")


## Step 2: train the model

`num_covariates=1` is the onpromotion channel, on top of the sales channel the
model always expects as channel 0. Other hyperparameters use the package
defaults. Training minimizes MSE in log space; validation reports NWRMSLE
(equal-weighted here, since an uploaded CSV has no perishable flags, so it is
plain RMSLE and not comparable to the Favorita benchmark figure).


In [ ]:
# 8. Model
model = MultiSeriesWaveNet(num_series=NUM_SERIES, horizon=HORIZON, num_covariates=1).to(device)
print(model.__class__.__name__, "| params:", sum(p.numel() for p in model.parameters()))


In [ ]:
# 9. Metric + training loop
def nwrmsle(y_true, y_pred, weights=None):
    y_true = np.clip(y_true, 0, None)
    y_pred = np.clip(y_pred, 0, None)
    sle = (np.log1p(y_pred) - np.log1p(y_true)) ** 2
    if weights is None:
        return float(np.sqrt(sle.mean()))
    w = weights.reshape(-1, 1)
    return float(np.sqrt((w * sle).sum() / (w * np.ones_like(sle)).sum()))

opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, nb = 0.0, 0
    for x, sid, y in train_loader:
        x, sid, y = x.to(device), sid.to(device), y.to(device)
        log_target = torch.log1p(y.clamp(min=0))
        opt.zero_grad()
        log_pred = model(x, sid)                 # (batch, HORIZON), log space
        loss = loss_fn(log_pred, log_target)
        loss.backward(); opt.step()
        running += loss.item(); nb += 1

    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for x, sid, y in val_loader:
            log_pred = model(x.to(device), sid.to(device))
            pred = torch.expm1(log_pred).clamp(min=0)
            yt.append(y.numpy()); yp.append(pred.cpu().numpy())
    if yt:
        val = nwrmsle(np.concatenate(yt), np.concatenate(yp))
        best_val = min(best_val, val)
        print(f"epoch {epoch:3d} | train MSE {running/max(nb,1):.4f} | val RMSLE {val:.4f}")
    else:
        print(f"epoch {epoch:3d} | train MSE {running/max(nb,1):.4f}")

print("Best val RMSLE:", round(best_val, 4),
      "(equal-weighted, your panel; not comparable to the Favorita ~0.61 figure)")


## Step 3: build the driver cards

The decomposition partitions the 90 input days into three groups: promotion days,
recent non-promotion days (the last 14), and everything else (typical). Starting
from a neutral baseline (flat mean-level sales, no promotion), it reveals the
groups cumulatively and measures the model's forecast, summed over
`REPORT_HORIZON` days in raw units, at each stage:

- **base** = baseline with the typical days revealed (the product's usual level).
- **trend** = the extra effect of revealing recent non-promotion days.
- **promo** = the extra effect of revealing the promotion days.

By construction `base + trend + promo == total_forecast` exactly.

This runs through the package's `CounterfactualExplainer` using the documented
API: `full_input` as a `(channels, timesteps)` array, per-channel
`baseline_values`, and `reveal_groups` as an OrderedDict of `(timesteps,)` bool
masks in baseline-first order. If the installed `explain()` behaves differently,
the cell falls back to an identical in-notebook computation and says so, so the
cards still build.


In [ ]:
# 10. Attribution: package path with an identical in-notebook fallback.
USE_PACKAGE_EXPLAINER = True

def _day_groups(promo_w, input_length, recent_days=RECENT_DAYS):
    promo_mask = promo_w > 0.5
    recent_mask = np.zeros(input_length, dtype=bool)
    recent_mask[-recent_days:] = True
    recent_non_promo = recent_mask & (~promo_mask)
    typical = (~promo_mask) & (~recent_mask)
    return typical, recent_non_promo, promo_mask

def _baseline_values(sales_history):
    return float(np.log1p(np.clip(sales_history, 0, None)).mean()), 0.0

def _inline_buckets(model, series_index_int, full_input_np, sales_baseline, promo_baseline,
                    typical, recent_non_promo, device, report_horizon=REPORT_HORIZON):
    """Exact sequential counterfactual masking (raw-unit sums over report_horizon)."""
    L = full_input_np.shape[-1]
    fi = torch.from_numpy(full_input_np).to(device)
    def predict(reveal_mask):
        x = fi.clone().unsqueeze(0)               # (1, 2, L)
        x[0, 0, :] = sales_baseline
        x[0, 1, :] = promo_baseline
        for t in np.where(reveal_mask)[0]:
            x[0, 0, t] = fi[0, t]
            x[0, 1, t] = fi[1, t]
        sid = torch.full((1,), series_index_int, dtype=torch.long, device=device)
        with torch.no_grad():
            log_pred = model(x, sid)
        return float(torch.expm1(log_pred.clamp(min=0))[0, :report_horizon].sum().cpu())
    p_typ  = predict(typical)
    p_tr   = predict(typical | recent_non_promo)
    p_full = predict(np.ones(L, dtype=bool))
    return {"base": p_typ, "trend": p_tr - p_typ, "promo": p_full - p_tr}, p_full

def _reduce_total(x, report_horizon=REPORT_HORIZON):
    """Sum a per-day contribution/prediction over the reported horizon.
    Robust to explain() returning per-day vectors or an already-reduced scalar."""
    a = np.asarray(x, dtype=float).ravel()
    return float(a[:report_horizon].sum()) if a.size > 1 else float(a.sum())

def _package_buckets(model, series_index_int, full_input_np, baseline_values,
                     typical, recent_non_promo, promo_mask, device):
    """Decomposition through wavexplain.CounterfactualExplainer, per the README API:
    (timesteps,) bool masks, per-channel baseline_values, full_input (channels, timesteps)."""
    L = full_input_np.shape[-1]
    def group_mask(days):
        m = np.zeros(L, dtype=bool); m[days] = True; return m
    reveal_groups = OrderedDict([                  # most baseline-like first
        ("base",  group_mask(typical)),
        ("trend", group_mask(recent_non_promo)),
        ("promo", group_mask(promo_mask)),
    ])
    explainer = CounterfactualExplainer(
        model, series_id=series_index_int, device=device,
        output_transform=lambda t: torch.expm1(t.clamp(min=0)),  # log-space -> raw units
    )
    contributions, baseline_pred, full_pred = explainer.explain(
        full_input_np, baseline_values=baseline_values, reveal_groups=reveal_groups,
    )
    base = _reduce_total(baseline_pred) + _reduce_total(contributions["base"])
    return {"base": base,
            "trend": _reduce_total(contributions["trend"]),
            "promo": _reduce_total(contributions["promo"])}, _reduce_total(full_pred)

_fallback_announced = {"done": False}
def attribute(model, arr, series_index_int, device):
    sales_w = arr["sales"][-INPUT_LENGTH:]
    promo_w = arr["promo"][-INPUT_LENGTH:]
    full_input_np = make_log_input(sales_w, promo_w).numpy()   # (2, L): log1p sales + raw promo
    sales_baseline, promo_baseline = _baseline_values(arr["sales"])
    typical, recent_non_promo, promo_mask = _day_groups(promo_w, INPUT_LENGTH)

    if USE_PACKAGE_EXPLAINER:
        try:
            return _package_buckets(model, series_index_int, full_input_np,
                                    [sales_baseline, promo_baseline],
                                    typical, recent_non_promo, promo_mask, device)
        except Exception as e:
            if not _fallback_announced["done"]:
                print("NOTE: package CounterfactualExplainer path failed "
                      f"({type(e).__name__}: {e}). Falling back to the identical "
                      "in-notebook computation; the numbers are the same.")
                _fallback_announced["done"] = True
    return _inline_buckets(model, series_index_int, full_input_np,
                           sales_baseline, promo_baseline,
                           typical, recent_non_promo, device)


In [ ]:
# 11. Forecast/history for the chart, explanation text, and the web-card schema
def forecast_and_history(model, arr, series_index_int, device):
    sales_w = arr["sales"][-INPUT_LENGTH:]
    promo_w = arr["promo"][-INPUT_LENGTH:]
    x = make_log_input(sales_w, promo_w).unsqueeze(0).to(device)
    sid = torch.full((1,), series_index_int, dtype=torch.long, device=device)
    with torch.no_grad():
        log_pred = model(x, sid)
    per_day = torch.expm1(log_pred.clamp(min=0))[0].cpu().numpy()   # (HORIZON,)
    return per_day, np.asarray(sales_w, dtype=float)

def explanation_for(is_promo, base, trend):
    if is_promo:
        return "Mostly explained by the active promotion, not a shift in underlying demand."
    if trend > max(abs(base) * 0.15, 1):
        return "Rising demand. Most of the forecast comes from a recent upward trend, not a promotion."
    if trend < -max(abs(base) * 0.15, 1):
        return "Softening demand. The forecast is below the usual pattern because of a recent downward trend."
    return "Follows this product's typical sales pattern."

def _split_store_item(sid):
    parts = str(sid).split("_")
    if len(parts) == 2 and all(p.lstrip("-").isdigit() for p in parts):
        return int(parts[0]), int(parts[1])
    return None, None

def build_web_card(model, sid, arr, series_index_int, device):
    buckets, total = attribute(model, arr, series_index_int, device)
    per_day, sales_w = forecast_and_history(model, arr, series_index_int, device)
    base, trend, promo = buckets["base"], buckets["trend"], buckets["promo"]
    max_abs = max(abs(base), abs(trend), abs(promo))
    is_promo = max_abs > 0 and abs(promo) == max_abs
    store, item = _split_store_item(sid)
    return {
        "series_id": str(sid),
        "store": store,
        "item": item,
        "promotion_driven": bool(is_promo),
        "horizon_label": f"next {REPORT_HORIZON} days",
        "total_forecast": round(total),
        "explanation": explanation_for(is_promo, base, trend),
        "drivers": [
            {"label": "Typical pattern for this product", "value": round(base),  "kind": "base"},
            {"label": "Active promotion",                 "value": round(promo), "kind": "promo"},
            {"label": "Recent trend",                     "value": round(trend), "kind": "trend"},
        ],
        "history":  [round(float(v), 1) for v in sales_w[-HISTORY_PLOTTED:]],
        "forecast": [round(float(v), 1) for v in per_day[:REPORT_HORIZON]],
    }


In [ ]:
# 12. Build cards for the highest-volume series and write cards.json
model.eval()
ranked = sorted(series_ids, key=lambda s: float(series[s]["sales"].sum()), reverse=True)
chosen = ranked[:MAX_CARDS]

cards = []
for sid in chosen:
    try:
        cards.append(build_web_card(model, sid, series[sid], id_to_index[sid], device))
        print("ok  ", sid)
    except Exception as e:
        print("skip", sid, ":", e)

with open("cards.json", "w") as f:
    json.dump(cards, f, indent=2)
print("Wrote cards.json with", len(cards), "cards.")
cards[0]


In [ ]:
# 13. Preview the cards inline (rendered right in the notebook output)
def chart_svg(history, forecast):
    W,Hh,xL,xR,yT,yB = 500,150,40,470,20,120
    h,n = len(history), len(history)+len(forecast)
    allv = history+forecast; lo,hi = min(allv),max(allv)
    if hi==lo: hi+=1; lo-=1
    xa = lambda i: xL+i*(xR-xL)/(n-1); ya = lambda v: yB-(v-lo)/(hi-lo)*(yB-yT)
    hp = ' '.join(f'{xa(i):.1f},{ya(v):.1f}' for i,v in enumerate(history))
    fi = [h-1]+[h+k for k in range(len(forecast))]; fv=[history[-1]]+forecast
    fp = ' '.join(f'{xa(i):.1f},{ya(v):.1f}' for i,v in zip(fi,fv))
    dv = (xa(h-1)+xa(h))/2
    return (f'<svg viewBox="0 0 {W} {Hh}" style="width:100%"><line x1="{dv:.1f}" y1="16" x2="{dv:.1f}" y2="124" stroke="#c9c7bd" stroke-dasharray="3 3"/>'
            f'<polyline fill="none" stroke="#888780" stroke-width="1.5" points="{hp}"/>'
            f'<polyline fill="none" stroke="#EF9F27" stroke-width="2" stroke-dasharray="5 4" points="{fp}"/></svg>')

def card_html(c):
    badge = ('#FAEEDA','#854F0B','Promotion-driven') if c['promotion_driven'] else ('#F1EFE8','#5F5E5A','Standard pattern')
    mx = max(abs(x['value']) for x in c['drivers']) or 1
    head = (f"Store {c['store']} &middot; Item {c['item']}"
            if c.get('store') is not None and c.get('item') is not None
            else f"Series {c['series_id']}")
    rows = ''
    for dvr in c['drivers']:
        col = '#EF9F27' if dvr['kind']=='promo' else '#888780'
        val = dvr['value'] if dvr['kind']=='base' else (f"+{dvr['value']}" if dvr['value']>=0 else dvr['value'])
        rows += (f"<div style='display:flex;justify-content:space-between;font-size:13px;margin:8px 0 4px'><span style='color:#5f5e5a'>{dvr['label']}</span><span>{val}</span></div>"
                 f"<div style='background:#f1efe8;border-radius:4px;height:8px'><div style='width:{max(2,round(abs(dvr['value'])/mx*100))}%;height:100%;border-radius:4px;background:{col}'></div></div>")
    return (f"<div style='background:#fff;border:1px solid #e0ded4;border-radius:12px;padding:20px 24px;max-width:420px;margin:16px auto;font-family:-apple-system,Arial,sans-serif'>"
            f"<div style='display:flex;justify-content:space-between'><span style='color:#5f5e5a;font-size:13px'>{head}</span>"
            f"<span style='background:{badge[0]};color:{badge[1]};font-size:12px;padding:3px 10px;border-radius:6px;font-weight:600'>{badge[2]}</span></div>"
            f"<p style='color:#5f5e5a;font-size:13px;margin:12px 0 2px'>Predicted sales, {c['horizon_label']}</p>"
            f"<p style='font-size:32px;font-weight:600;margin:0'>{c['total_forecast']} units</p>{chart_svg(c['history'],c['forecast'])}"
            f"<p style='font-size:14px;color:#444;margin:8px 0 12px'>{c['explanation']}</p>{rows}</div>")

from IPython.display import HTML, display
display(HTML(''.join(card_html(c) for c in cards)))


In [ ]:
# 14. Render a standalone dashboard.html (the same look as the demo page)
_PAGE_CSS = """
  body { font-family: -apple-system, Helvetica, Arial, sans-serif; background: #f1efe8; padding: 40px; }
  .card { background: #ffffff; border-radius: 12px; border: 1px solid #e0ded4; padding: 24px 28px; max-width: 420px; margin: 0 auto 24px; }
  .header { display: flex; justify-content: space-between; align-items: flex-start; margin-bottom: 4px; }
  .muted { color: #5f5e5a; font-size: 13px; }
  .badge { font-size: 12px; padding: 3px 10px; border-radius: 6px; font-weight: 600; white-space: nowrap; }
  .big-number { font-size: 32px; font-weight: 600; margin: 4px 0; }
  .chart { width: 100%; height: auto; display: block; margin: 8px 0; }
  .explanation { font-size: 14px; color: #444441; margin-bottom: 20px; }
  .section-title { font-size: 13px; color: #5f5e5a; font-weight: 600; margin-bottom: 12px; }
  .row { margin-bottom: 10px; }
  .row-label { display: flex; justify-content: space-between; font-size: 13px; margin-bottom: 4px; }
  .bar-track { background: #f1efe8; border-radius: 4px; height: 8px; overflow: hidden; }
  .bar-fill { height: 100%; border-radius: 4px; }
  .total-row { display: flex; justify-content: space-between; align-items: center; margin-top: 16px; padding-top: 12px; border-top: 1px solid #e0ded4; }
  .footnote { font-size: 12px; color: #888780; margin-top: 16px; }
"""

def build_chart_svg(history, forecast):
    W, H, xL, xR, yT, yB = 500, 150, 40, 470, 20, 120
    h, n = len(history), len(history) + len(forecast)
    if h == 0 or n < 2:
        return ""
    allv = list(history) + list(forecast)
    lo, hi = min(allv), max(allv)
    if hi == lo:
        hi += 1; lo -= 1
    def x_at(i): return xL + i * (xR - xL) / (n - 1)
    def y_at(v): return yB - (v - lo) / (hi - lo) * (yB - yT)
    hist_pts = " ".join(f"{x_at(i):.1f},{y_at(v):.1f}" for i, v in enumerate(history))
    fc_idx = [h - 1] + [h + k for k in range(len(forecast))]
    fc_vals = [history[-1]] + list(forecast)
    fc_pts = " ".join(f"{x_at(i):.1f},{y_at(v):.1f}" for i, v in zip(fc_idx, fc_vals))
    dots = "".join(f'<circle cx="{x_at(h + k):.1f}" cy="{y_at(v):.1f}" r="2.5" fill="#EF9F27"/>'
                   for k, v in enumerate(forecast))
    div_x = (x_at(h - 1) + x_at(h)) / 2
    return f"""<svg class="chart" viewBox="0 0 {W} {H}" role="img" aria-label="Sales history and forecast">
    <line x1="{xL}" y1="{yB + 4}" x2="{xR}" y2="{yB + 4}" stroke="#e0ded4" stroke-width="1"/>
    <line x1="{div_x:.1f}" y1="{yT - 4}" x2="{div_x:.1f}" y2="{yB + 4}" stroke="#c9c7bd" stroke-width="1" stroke-dasharray="3 3"/>
    <polyline fill="none" stroke="#888780" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round" points="{hist_pts}"/>
    <polyline fill="none" stroke="#EF9F27" stroke-width="2" stroke-dasharray="5 4" stroke-linecap="round" stroke-linejoin="round" points="{fc_pts}"/>
    {dots}
    <text x="{xL}" y="{H - 4}" font-size="11" fill="#888780">recent history</text>
    <text x="{xR}" y="{H - 4}" font-size="11" fill="#BA7517" text-anchor="end">forecast</text>
  </svg>"""

def _card_body_html(c):
    is_promo = c["promotion_driven"]
    badge_label = "Promotion-driven" if is_promo else "Standard pattern"
    badge_bg = "#FAEEDA" if is_promo else "#F1EFE8"
    badge_text = "#854F0B" if is_promo else "#5F5E5A"
    dvals = {d["kind"]: d["value"] for d in c["drivers"]}
    typical, promo, trend = dvals["base"], dvals["promo"], dvals["trend"]
    total = c["total_forecast"]
    max_bucket = max(abs(typical), abs(promo), abs(trend), 1)
    def bar_width(v): return max(2, round(abs(v) / max_bucket * 100))
    if c.get("store") is not None and c.get("item") is not None:
        head = f"Store {c['store']} &middot; Item {c['item']}"
    else:
        head = f"Series {c['series_id']}"
    chart = build_chart_svg(c.get("history", []), c.get("forecast", []))
    return f"""<div class="card">
  <div class="header">
    <p class="muted">{head}</p>
    <span class="badge" style="background:{badge_bg}; color:{badge_text};">{badge_label}</span>
  </div>
  <p class="muted">Predicted sales, {c['horizon_label']}</p>
  <p class="big-number">{total} units</p>
  {chart}
  <p class="explanation">{c['explanation']}</p>
  <p class="section-title">What's driving this number</p>
  <div class="row">
    <div class="row-label"><span class="muted">Typical pattern for this product</span><span>{typical}</span></div>
    <div class="bar-track"><div class="bar-fill" style="width:{bar_width(typical)}%; background:#888780;"></div></div>
  </div>
  <div class="row">
    <div class="row-label"><span class="muted">Active promotion</span><span style="color:{badge_text}">{'+' if promo >= 0 else ''}{promo}</span></div>
    <div class="bar-track"><div class="bar-fill" style="width:{bar_width(promo)}%; background:#EF9F27;"></div></div>
  </div>
  <div class="row">
    <div class="row-label"><span class="muted">Recent trend</span><span>{'+' if trend >= 0 else ''}{trend}</span></div>
    <div class="bar-track"><div class="bar-fill" style="width:{bar_width(trend)}%; background:#888780;"></div></div>
  </div>
  <div class="total-row">
    <span class="muted">Total forecast</span>
    <span style="font-weight:600; font-size:16px;">{total} units</span>
  </div>
  <p class="footnote">Based on {INPUT_LENGTH} days of sales history. Breakdown is model-generated and may not capture every factor.</p>
</div>"""

def render_dashboard_html(cards, output_path="dashboard.html"):
    body = "\n".join(_card_body_html(c) for c in cards)
    html = f"<!DOCTYPE html><html><head><meta charset=\"utf-8\"><style>{_PAGE_CSS}</style></head><body>{body}</body></html>"
    with open(output_path, "w") as f:
        f.write(html)
    print(f"Saved {output_path} with {len(cards)} cards.")

render_dashboard_html(cards, "dashboard.html")


In [ ]:
# 15. Sanity check + download
for c in cards[:3]:
    flag = "PROMO" if c["promotion_driven"] else ""
    print(f"[{c['series_id']}] total {c['total_forecast']} ({c['horizon_label']}) {flag}")
    for d in c["drivers"]:
        print(f"   {d['label']:<34} {d['value']:>6}  ({d['kind']})")
    s = sum(d["value"] for d in c["drivers"])
    if s != c["total_forecast"]:
        print(f"   (drivers sum {s} vs total {c['total_forecast']}: <=1-unit rounding gap is expected)")

try:
    from google.colab import files as _files
    _files.download("cards.json")
    _files.download("dashboard.html")
except Exception:
    print("Not in Colab, or download blocked. Grab cards.json and dashboard.html "
          "from the file browser on the left.")


## Notes and honest limitations

- **Training is per-run and per-your-series.** The model does not transfer to
  series it has not seen; that is the per-series embedding, not a bug.
- **The RMSLE here is on your panel** and is not comparable to the Favorita
  benchmark. Use it to watch training, not as a headline number.
- **The `Promotion-driven` badge is provisional.** The promotion bucket is the
  least-validated part of the decomposition, so do not present the badge as a
  strong claim until it is validated at scale.
- **Counterfactual, not causal.** Contributions describe how the model's forecast
  changes as inputs are revealed on a baseline, not real-world cause and effect.
- **Rounding.** Buckets and the total are rounded independently, so the three
  driver values can differ from the total by at most one unit. The underlying
  decomposition sums exactly.
- **Package vs fallback.** Card cell 10 uses the package's
  `CounterfactualExplainer` (README API); if the installed `explain()` behaves
  differently it falls back to an identical in-notebook computation and prints a
  note. Both produce the same numbers.
